In [19]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu

In [20]:
problems = {
    "regression": ["nguyen_9_dense", "nguyen_10_dense", "feynman_13", "feynman_43"],
    "multiregression": ["feynman_13_61", "feynman_42_43_48", "feynman_13_61_62_70_77_100", "composite_function"],
    "control": ["inverted_double_pendulum", "reacher", "swimmer", "hopper", "walker2d", "halfcheetah"]
}
seeds = range(0, 10)
algorithms = ["ga", "gomea"]
solvers = ["cgp", "lgp"]

In [22]:
missing = 0
dfs = []
for algorithm in algorithms:
    for solver in solvers:
        for problem_group in problems.keys():
            for problem in problems[problem_group]:
                for pop in ["small", "large"]:
                    div = "_large_pop" if pop == "large" else ("_diversity" if pop == "diverse" else "")
                    for seed in seeds:
                        if algorithm == "gomea":
                            for fos in ["U", "RT", "LT"]:  #, "LT1"]:
                                tmp_df = pd.read_csv(
                                    f"../results/{problem_group.replace('multi', '')}/{algorithm}_{solver}_{problem}_{fos}_{seed}_fi{div}.csv")
                                tmp_df["seed"] = seed
                                tmp_df["pop"] = pop
                                tmp_df["algorithm"] = f"{algorithm}-{fos}"
                                tmp_df["group"] = problem_group
                                tmp_df["problem"] = (problem.replace("13_61_62_70_77_100", "c")
                                                     .replace("42_43_48", "b").replace("13_61", "a").replace("_dense",
                                                                                                             ""))
                                tmp_df["solver"] = solver
                                dfs.append(tmp_df)
                        else:
                            tmp_df = pd.read_csv(
                                f"../results/{problem_group.replace('multi', '')}/{algorithm}_{solver}_{problem}_{seed}{div}.csv")
                            tmp_df["seed"] = seed
                            tmp_df["algorithm"] = algorithm
                            tmp_df["problem"] = (problem.replace("13_61_62_70_77_100", "c")
                                                 .replace("42_43_48", "b").replace("13_61", "a").replace("_dense", ""))
                            tmp_df["solver"] = solver
                            tmp_df["group"] = problem_group
                            tmp_df["pop"] = pop
                            dfs.append(tmp_df)
df = pd.concat(dfs, ignore_index=True)
df = df.replace(-np.inf, -1e6)
df.loc[(df["group"] == "control") & (df["test_accuracy"].isna()), "test_accuracy"] = df["fitness"]
df["max_eval"] = df.groupby(["algorithm", "solver", "problem", "group", "seed", "pop"])[
    "evaluation"].transform("max")
df = df[df["evaluation"] == df["max_eval"]].drop(columns="max_eval")

In [23]:
for problem in df.problem.unique():
    for solver in df.solver.unique():
        for pop in df["pop"].unique():
            tmp_df = df[(df["problem"] == problem) & (df["solver"] == solver) & (df["pop"] == pop)]
            df_wide = tmp_df.pivot(index="seed", columns="algorithm", values="fitness").reset_index()[
                ["ga", "gomea-U", "gomea-RT", "gomea-LT"]]
            group = tmp_df["group"].tolist()[0]
            df_wide.to_csv(f"../pgfplots/{group}_{problem}_{solver}_{pop}.txt", index=False, sep="\t")